In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%pip install lightgbm
%reload_ext autoreload
import joblib
import re
import sys
user_name = spark.sql("SELECT current_user()").collect()[0][0]
sys.path.insert(0, f"/Workspace/Users/{user_name}/EV-Charging-Demand-Optimisation/energy-forecasting")
import lightgbm as lgb
import pandas as pd
from src.models.forecasting.trainer import train_quantile_lgbm, train_lgbm_no_mlflow
from sklearn.metrics import mean_absolute_error
import mlflow
import numpy as np
import pyspark.sql.functions as F

In [0]:
df_gold = spark.read.table("gold_carbon_weather_regional")
print(f"Rows: {df_gold.count()}")
print(f"Columns: {len(df_gold.columns)}") 

In [0]:
# Drop rows with nulls - the first 336 warm up period

df_train = df_gold.dropna(subset=["lag_1", "lag_2", "lag_48", "lag_336"]) 
print(f"Rows after dropping NULL lags: {df_train.count()}")

In [0]:
def train_for_one_region(regional_df: pd.DataFrame):
    """Here's what the function needs to do:
    1. Take a pd.DataFrame for one region
    2. Define features X and target y
    3. Train a LightGBM model
    4. Return a summary DataFrame with region_name, region_id, and mae"""
    region_name = regional_df["region_name"].iloc[0]
    region_id = regional_df["region_id"].iloc[0]
    safe_region = re.sub(r"[^a-z0-9]", "_", region_name.lower())

    feature_cols = ["lag_1", "lag_2", "lag_48", "lag_336", "rolling_avg_7day", "temperature", "wind_speed", "radiation","temp_roll_avg_7day"]
    X = regional_df[feature_cols]
    y = regional_df["carbon_intensity"]
    
    alphas = [0.10, 0.50, 0.90]
    
    summary_metrics = []
    for alpha in alphas:
        model, oof_preds, pb_loss = train_lgbm_no_mlflow(X, y, alpha)
        # oof_preds will contain nans
        mask = ~np.isnan(oof_preds)
        mae = mean_absolute_error(y_true=y[mask], y_pred=oof_preds[mask])
        
        summary_metrics.append({
            "region": region_name,
            "region_id": region_id,
            "alpha": alpha,
            "mae": mae,
            "pinball_loss": pb_loss
        })

        # Save the model to volumes in the workspace
        path = f"/Volumes/workspace/default/models/lgbm_p{int(alpha*100)}_{safe_region}.pkl"

        joblib.dump(model, path)    
    
    return pd.DataFrame(summary_metrics)

In [0]:
# Set to False to skip retraining and load from saved results
retrain = True

In [0]:
if retrain:
    all_regions_summary_df = df_train.groupBy(F.col("region_name")).applyInPandas(train_for_one_region, schema="region string, region_id long, alpha double, mae double, pinball_loss double")
else:
    print("Skipping retraining - loading from saved CSV")
    all_regions_summary_df = spark.read.option("header", "true").schema("region string, region_id long, alpha double, mae double, pinball_loss double").csv("/Volumes/workspace/default/models/all_regions_summary_df.csv")


In [0]:
# Write to csv 
all_regions_summary_df.write.mode("overwrite").option("header", "true").csv("/Volumes/workspace/default/models/all_regions_summary_df.csv")
display(all_regions_summary_df)

# Now I want to load from csv to re-create all_regions_summary_df
# Specify schema to preserve data types
all_regions_summary_df = spark.read.option("header", "true").schema("region string, region_id long, alpha double, mae double, pinball_loss double").csv("/Volumes/workspace/default/models/all_regions_summary_df.csv")
display(all_regions_summary_df)

In [0]:
results_pdf.to_csv("/Volumes/workspace/default/models/08_train_regional_models_with_weather.csv", index=False)
print("Results saved to /Volumes/workspace/default/models/08_train_regional_models_with_weather.csv")

In [0]:
# Analyze results to find best models

# Convert to pandas for easier analysis
results_pdf = all_regions_summary_df.toPandas()


In [0]:
# Analyze results to find best models

# Convert to pandas for easier analysis
results_pdf = all_regions_summary_df.toPandas()

print("\n=== Best models by MAE (lower is better) ===")
print(results_pdf.sort_values('mae').head(10))

print("\n=== Best models by Pinball Loss (lower is better) ===")
print(results_pdf.sort_values('pinball_loss').head(10))

print("\n=== Average metrics by alpha (quantile) ===")
print(results_pdf.groupby('alpha')[['mae', 'pinball_loss']].mean())

print("\n=== Average metrics by region ===")
print(results_pdf.groupby('region')[['mae', 'pinball_loss']].mean().sort_values('mae'))